# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [7]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* <br>A line chart is appropriate because it shows population growth over time and allows comparison between continents. Africa is highlighted in blue, while other continents are grey for context.

In [10]:
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

baseline = (
    continent_pop[continent_pop['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Population_1970'})
)

continent_pop = continent_pop.merge(baseline, on='Continent')

continent_pop['Growth Index'] = (
    continent_pop['Population'] /
    continent_pop['Population_1970'] * 100
)

In [12]:
colors = {
    'Africa': '#2878B5',
    'Asia': '#A7ADB4',
    'Europe': '#A7ADB4',
    'North America': '#A7ADB4',
    'South America': '#A7ADB4',
    'Oceania': '#A7ADB4'
}

fig = px.line(
    continent_pop,
    x='Year',
    y='Growth Index',
    color='Continent',
    color_discrete_map=colors,
    markers=True,
    title='Africa’s population has grown fastest since 1970',
    labels={
        'Year': '',
        'Growth Index': 'Population index',
        'Continent': ''
    }
)

fig.update_traces(line=dict(width=2), marker=dict(size=5))

fig.update_traces(
    selector=dict(name='Africa'),
    line=dict(width=4),
    marker=dict(size=8)
)

fig.update_layout(
    template='plotly_white',
    height=550,
    showlegend=True,
    title=dict(x=0.05),
    margin=dict(l=60, r=40, t=80, b=50)
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(
    showgrid=True,
    gridcolor='lightgrey'
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* <br> I chose a horizontal bar chart to compare the 15 countries with the highest population increases between 1970 and 2022. Countries are ranked in descending order, highlighting India's and China's dominance. African countries are highlighted in blue while others are grey, making the comparison clear.

In [14]:
df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = df.nlargest(15, 'Population Added').copy()

africa_count = (top15['Continent'] == 'Africa').sum()

print("Number of African countries:", africa_count)

top15[['Country', 'Continent', 'Population Added']]

Number of African countries: 4


,Country,Continent,Population Added
92,India,Asia,859671872
41,China,Asia,603352887
156,Pakistan,Asia,176533990
149,Nigeria,Africa,162971948
93,Indonesia,Asia,160272945
221,United States,North America,137961517
27,Brazil,South America,118943623
16,Bangladesh,Asia,103644512
63,Ethiopia,Africa,95071678
55,DR Congo,Africa,78858479


In [19]:
top15 = top15.sort_values(
    'Population Added',
    ascending=False
)


top15['Colour'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    orientation='h',
    color='Colour',
    color_discrete_map={
        'Africa': '#2878B5',
        'Other continents': '#A7ADB4'
    },
    title='India and China Accounted for the Largest Population Gains (1970–2022)',
    labels={
        'Population Added': 'Population added (millions)',
        'Country': '',
        'Colour': ''
    },
    text=top15['Population Added'] / 1_000_000
)

fig.update_traces(
    texttemplate='%{text:.1f}M',
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    template='plotly_white',
    showlegend=False,
    height=650,
    margin=dict(l=130, r=90, t=80, b=60)
)

fig.update_xaxes(
    tickformat='~s',
    ticksuffix='',
    showgrid=False,
    title='Population added (people)'
)

fig.update_yaxes(
    autorange='reversed',
    showgrid=False
)

fig.show()



---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*<br> I chose a scatter plot to examine the relationship between land area and population in 2022. Different colours represent continents, while logarithmic scales make comparisons easier. Selected countries are labelled to highlight those that break the general pattern.

In [30]:
fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    log_x=True,
    log_y=True,
    title='Does a Larger Land Area Mean a Larger Population? (2022)',
    labels={
        'Area': 'Land area (km²)',
        '2022 Population': 'Population (2022)',
        'Continent': ''
    }
)

countries = ['India', 'China', 'Bangladesh', 'Russia', 'Canada']

for country in countries:
    row = df[df['Country'] == country].iloc[0]
    fig.add_annotation(
        x=row['Area'],
        y=row['2022 Population'],
        text=country,
        showarrow=True,
        ax=20,
        ay=-20
    )

fig.update_layout(
    template='plotly_white',
    height=650
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* <br> I chose a box plot to compare population density across continents. It shows the spread, median and outliers, while individual points represent countries.

In [32]:
fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    points='all',
    hover_name='Country',
    title='Population Density Across Continents (2022)',
    labels={
        'Continent': '',
        'Density': 'People per km²'
    }
)

fig.update_layout(
    template='plotly_white',
    showlegend=False,
    height=550
)

fig.update_yaxes(type='log')

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* <br> I chose a choropleth map to show geographical patterns in population change. Colours distinguish countries with population growth from those experiencing decline, while hover labels show the percentage change.

In [25]:
df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

fig = px.choropleth(
    df,
    locations='CCA3',
    color='Population Change (%)',
    hover_name='Country',
    color_continuous_scale=[
        [0, '#B2182B'],
        [0.5, '#F5F5F5'],
        [1, '#2166AC']
    ],
    color_continuous_midpoint=0,
    title='Population Growth and Decline Across the World (2000–2022)',
    labels={
        'Population Change (%)': 'Population change (%)'
    }
)

fig.update_layout(
    template='plotly_white',
    height=550
)

fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* <br> I chose a treemap to show population distribution at two levels: continents and individual countries. The size of each rectangle represents its population, making it easy to compare their relative contributions to the world's population.

In [34]:
asia_share = (
    df[df['Continent'] == 'Asia']['2022 Population'].sum()
    / df['2022 Population'].sum() * 100
)

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    title=f'Asia alone Accounts for {asia_share:.0f}% of the World’s Population (2022)',
    color='Continent',
    hover_data={'2022 Population': ':,'}
)

fig.update_layout(
    template='plotly_white',
    height=650
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* <br> I chose a line chart to show how population shares changed over time. It makes it easy to compare Africa, Asia, Europe and the rest of the world, highlighting which regions gained or lost their share between 1970 and 2022.

In [27]:
df_long['Region'] = df_long['Continent'].apply(
    lambda x: x if x in ['Africa', 'Asia', 'Europe']
    else 'Rest of the world'
)

share = (
    df_long.groupby(['Region', 'Year'])['Population']
    .sum()
    .reset_index()
)

share['World Share (%)'] = (
    share['Population'] /
    share.groupby('Year')['Population'].transform('sum') * 100
)

fig = px.line(
    share,
    x='Year',
    y='World Share (%)',
    color='Region',
    markers=True,
    title='Changing Shares of World Population (1970–2022)',
    labels={
        'Year': '',
        'World Share (%)': 'Share of world population (%)',
        'Region': ''
    }
)

fig.update_layout(
    template='plotly_white',
    height=550
)

fig.show()
